### Converting mask into usable folder

### Masks Tensor shape: (num_objects, height, width)

Example: (3, 160, 160) 👉 3 potholes → 3 masks

### STEP 3: Resize Mask (IMPORTANT)

YOLO masks are often resized internally, so match original image:

## FULL WORKING CODE [for masks extraction]

masks = r.masks.data

This is a PyTorch tensor, typically like: Tensor shape: (num_objects, height, width)

Example: (3, 160, 160)

3 potholes → 3 masks

### STEP 4: Visualize Mask (VERY USEFUL)

### STEP 5: Extract Area from Mask (Connect to Previous Step)


In [12]:
from ultralytics import YOLO
import cv2
import numpy as np

# Load model ONCE (important)
model = YOLO("C:/Users/vaibh/Downloads/runs/segment/train/weights/best.pt")


def get_mask_from_model(image):
    """
    Run YOLO segmentation and return largest pothole mask.
    
    Args:
        image (np.array): input image (BGR)
    
    Returns:
        mask_np (np.array): binary mask (0 or 1)
    """

    results = model(image)

    for r in results:

        # No detection case
        if r.masks is None:
            return None

        masks = r.masks.data  # tensor (N, H, W)

        best_mask = None
        max_area = 0

        for mask in masks:
            mask_np = mask.cpu().numpy()

            # Resize to original image size
            mask_np = cv2.resize(mask_np, (image.shape[1], image.shape[0]))

            # Convert to binary
            mask_np = (mask_np > 0.5).astype(np.uint8)

            # Compute area
            area = np.sum(mask_np)

            # Select largest pothole
            if area > max_area:
                max_area = area
                best_mask = mask_np

        return best_mask

### Depth Estimation

#### METHOD 1: Intensity-Based Depth (Best Starting Point)
* Deeper pothole → darker interior
* Because less light reflects

| Avg Intensity | Depth Score    |
| ------------- | -------------- |
| 200 (bright)  | 0.21 → shallow |
| 100 (medium)  | 0.60 → medium  |
| 40 (dark)     | 0.84 → deep    |

#### METHOD 2: Edge/Gradient-Based Depth (Improves Accuracy)

* Deeper potholes → sharper edges
* More gradient variation

### FINAL DEPTH FUNCTION (READY TO USE)

In [6]:
import cv2
import numpy as np

def estimate_depth(image, mask_np):
    """
    Estimate relative depth of pothole using intensity + edge features.

    Args:
        image (np.array): Original BGR image (cv2 format)
        mask_np (np.array): Binary mask (0 or 1)

    Returns:
        float: depth score (0 to 1)
    """

    # Safety check
    if mask_np.sum() == 0:
        return 0.0

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # --- 1. Intensity-based depth ---
    pothole_pixels = gray[mask_np == 1]

    avg_intensity = np.mean(pothole_pixels)

    # Normalize (darker = deeper)
    intensity_depth = 1 - (avg_intensity / 255.0)

    # --- 2. Edge-based depth ---
    edges = cv2.Canny(gray, 50, 150)

    edge_pixels = edges[mask_np == 1]

    # Normalize edge density
    edge_density = np.sum(edge_pixels) / (mask_np.sum() * 255.0 + 1e-6)

    # --- 3. Combine both ---
    depth_score = 0.6 * intensity_depth + 0.4 * edge_density

    # Clamp between 0 and 1
    depth_score = np.clip(depth_score, 0, 1)

    return float(depth_score)

### Final Area Function

#### AREA FUNCTION (Production Ready)

This uses:

* mask (primary, accurate)
* optional box (fallback / hybrid)

In [7]:
import numpy as np

def calculate_area(mask_np, box=None, img_shape=None):
    """
    Calculate normalized area of pothole.

    Args:
        mask_np (np.array): Binary mask (0 or 1)
        box (tuple, optional): (x1, y1, x2, y2)
        img_shape (tuple, optional): (H, W)

    Returns:
        float: normalized area (0 to 1)
    """

    # --- Safety check ---
    if mask_np is None or mask_np.sum() == 0:
        return 0.0

    # --- 1. Mask-based area (primary) ---
    total_pixels = mask_np.shape[0] * mask_np.shape[1]
    mask_area = np.sum(mask_np) / total_pixels

    # --- 2. Optional box-based area ---
    if box is not None and img_shape is not None:
        x1, y1, x2, y2 = box
        box_pixels = (x2 - x1) * (y2 - y1)
        box_area = box_pixels / (img_shape[0] * img_shape[1])

        # Combine both (mask is more reliable)
        final_area = 0.7 * mask_area + 0.3 * box_area
    else:
        final_area = mask_area

    return float(np.clip(final_area, 0, 1))

### SEVERITY FUNCTION [Model Output]

Now combine:

* area
* depth

In [8]:
def calculate_severity(area, depth):
    """
    Calculate severity of pothole using area + depth.

    Args:
        area (float): normalized area (0 to 1)
        depth (float): depth score (0 to 1)

    Returns:
        float: severity score (0 to 1)
    """

    # Weighted combination
    severity = 0.6 * area + 0.4 * depth

    # Clamp to safe range
    severity = np.clip(severity, 0, 1)

    return float(severity)

### Priority Function

Priority function should:

Accept raw inputs from pipeline
Be modular (easy to improve later)
Work with or without APIs (fallback support)

In [9]:
import numpy as np
from datetime import datetime

# --- Helper functions ---

def get_traffic_score(road_type=None, traffic_api_score=None):
    if traffic_api_score is not None:
        return np.clip(traffic_api_score, 0, 1)

    # fallback
    traffic_map = {
        "highway": 1.0,
        "main_road": 0.8,
        "street": 0.5
    }
    return traffic_map.get(road_type, 0.5)


def get_location_score(location_type=None, places_score=None):
    if places_score is not None:
        return np.clip(places_score, 0, 1)

    # fallback
    location_map = {
        "hospital": 1.0,
        "school": 0.9,
        "market": 0.7,
        "residential": 0.5
    }
    return location_map.get(location_type, 0.5)


def get_time_score(created_at):
    if created_at is None:
        return 0.0

    days = (datetime.now() - created_at).days
    return np.clip(days / 10.0, 0, 1)


# --- MAIN PRIORITY FUNCTION ---

def calculate_priority_pipeline(
    severity,
    road_type=None,
    location_type=None,
    created_at=None,
    traffic_api_score=None,
    places_score=None
):
    """
    Full priority function integrated with pipeline.

    Args:
        severity (float): from your model (0–1)
        road_type (str): fallback traffic type
        location_type (str): fallback location type
        created_at (datetime): report timestamp
        traffic_api_score (float): optional real traffic score
        places_score (float): optional location importance from API

    Returns:
        dict: priority results
    """

    # --- Step 1: get scores ---
    traffic = get_traffic_score(road_type, traffic_api_score)
    location = get_location_score(location_type, places_score)
    time = get_time_score(created_at)

    # --- Step 2: weighted priority ---
    priority = (
        0.4 * severity +
        0.3 * traffic +
        0.2 * location +
        0.1 * time
    )

    priority = float(np.clip(priority, 0, 1))

    # --- Step 3: label ---
    if priority < 0.4:
        label = "LOW"
    elif priority < 0.7:
        label = "MEDIUM"
    else:
        label = "HIGH"

    return {
        "priority_score": priority,
        "priority_label": label,
        "components": {
            "severity": severity,
            "traffic": traffic,
            "location": location,
            "time": time
        }
    }

### CONNECT EVERYTHING INTO ONE PIPELINE (MANDATORY)

In [10]:
def process_pothole(image, mask_np, lat, lon, created_at):
    
    # 1. Area
    area = calculate_area(mask_np)

    # 2. Depth
    depth = estimate_depth(image, mask_np)

    # 3. Severity
    severity = calculate_severity(area, depth)

    # 4. Map features (use fallback for now)
    road_type = "main_road"
    location_type = "residential"

    # 5. Priority
    priority_data = calculate_priority_pipeline(
        severity,
        road_type=road_type,
        location_type=location_type,
        created_at=created_at
    )

    return {
        "area": area,
        "depth": depth,
        "severity": severity,
        **priority_data     # unpack dictionary into another dictionary 
    }

### Expected Output:

{
    "area": 0.18,
    "depth": 0.72,
    "severity": 0.43,
    "priority_score": 0.78,
    "priority_label": "HIGH"
}

### Backend Flow:

* Frontend (user uploads image)
        ↓
* Backend API (FastAPI)
        ↓
* YOLO model runs
        ↓
* process_pothole()
        ↓
* Return JSON response

run --> fastapi dev main.py

In [ ]:
# def process_video(video_path, save_output=False, output_path="output.mp4"):

#     import cv2
#     import numpy as np
#     from datetime import datetime

#     cap = cv2.VideoCapture(video_path)

#     if not cap.isOpened():
#         print("Error opening video")
#         return

#     # --- Optional video writer ---
#     if save_output:
#         fourcc = cv2.VideoWriter_fourcc(*'mp4v')
#         out = cv2.VideoWriter(
#             output_path,
#             fourcc,
#             cap.get(cv2.CAP_PROP_FPS),
#             (int(cap.get(3)), int(cap.get(4)))
#         )

#     while True:
#         ret, frame = cap.read()
#         if not ret:
#             break

#         # --- YOLO inference ---
#         results = model(frame)

#         for r in results:
#             if r.masks is None:
#                 continue

#             masks = r.masks.data

#             for mask in masks:
#                 # --- Convert mask ---
#                 mask_np = mask.cpu().numpy()

#                 # Resize properly
#                 mask_np = cv2.resize(
#                     mask_np,
#                     (frame.shape[1], frame.shape[0]),
#                     interpolation=cv2.INTER_LINEAR
#                 )

#                 # Smooth edges
#                 mask_np = cv2.GaussianBlur(mask_np, (5, 5), 0)

#                 # Threshold
#                 mask_np = (mask_np > 0.5).astype(np.uint8)

#                 # --- Morphological cleaning ---
#                 kernel = np.ones((3, 3), np.uint8)

#                 mask_np = cv2.morphologyEx(mask_np, cv2.MORPH_OPEN, kernel)
#                 mask_np = cv2.morphologyEx(mask_np, cv2.MORPH_CLOSE, kernel)

#                 # --- Keep largest contour (remove noise blobs) ---
#                 contours, _ = cv2.findContours(
#                     mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
#                 )

#                 if contours:
#                     largest = max(contours, key=cv2.contourArea)
#                     clean_mask = np.zeros_like(mask_np)
#                     cv2.drawContours(clean_mask, [largest], -1, 1, -1)
#                     mask_np = clean_mask

#                 # --- Pipeline ---
#                 area = calculate_area(mask_np)
#                 depth = estimate_depth(frame, mask_np)
#                 severity = calculate_severity(area, depth)

#                 priority_data = calculate_priority_pipeline(
#                     severity,
#                     road_type="main_road",
#                     location_type="residential",
#                     created_at=datetime.now()
#                 )

#                 # --- Overlay mask (smooth & clean) ---
#                 overlay = frame.copy()
#                 overlay[mask_np == 1] = [0, 0, 255]

#                 alpha = 0.4
#                 frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

#                 # --- Draw contour ---
#                 cv2.drawContours(frame, [largest], -1, (0, 255, 0), 2)

#                 # --- Display info ---
#                 text = f"P:{priority_data['priority_label']} | S:{severity:.2f}"
#                 cv2.putText(
#                     frame,
#                     text,
#                     (30, 50),
#                     cv2.FONT_HERSHEY_SIMPLEX,
#                     1,
#                     (0, 255, 0),
#                     2
#                 )

#         # --- Show frame ---
#         cv2.imshow("Pothole Detection", frame)

#         # --- Save frame ---
#         if save_output:
#             out.write(frame)

#         # Press 'q' to exit
#         if cv2.waitKey(1) & 0xFF == ord('q'):
#             break

#     cap.release()

#     if save_output:
#         out.release()

#     cv2.destroyAllWindows()

In [25]:
process_video("C:/Projects/AI and ML/Deep Learning/Projects/Pothole Detection/Pothole_Segmentation_YOLOv8/sample_video.mp4")

0: 384x640 6 Potholes, 57.5ms
Speed: 5.8ms preprocess, 57.5ms inference, 4.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Potholes, 31.5ms
Speed: 2.6ms preprocess, 31.5ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Potholes, 14.1ms
Speed: 1.9ms preprocess, 14.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Potholes, 15.3ms
Speed: 1.4ms preprocess, 15.3ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Potholes, 14.2ms
Speed: 1.4ms preprocess, 14.2ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Potholes, 12.3ms
Speed: 1.5ms preprocess, 12.3ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 Potholes, 11.8ms
Speed: 2.6ms preprocess, 11.8ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Potholes, 11.5ms
Speed: 1.1ms preprocess, 11.5ms inference, 2.1ms postprocess per image at 